In [1]:
import torch
from datasets import Dataset
from modelscope import snapshot_download, AutoTokenizer
from swanlab.integration.transformers import SwanLabCallback
from qwen_vl_utils import process_vision_info
from peft import LoraConfig, TaskType, get_peft_model, PeftModel
from transformers import (
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq,
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor,
)
import swanlab
import json

D:\toolkit\anaconda\envs\llm\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# # 在modelscope上下载Qwen2.5-VL模型到本地目录下
cache_dir = "D:/cache/huggingface"
model_path = "Qwen/Qwen2.5-VL-7B-Instruct"
# 使用Transformers加载模型权重
tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=False, trust_remote_code=True, cache_dir=cache_dir+"/models")
processor = AutoProcessor.from_pretrained(model_path, cache_dir=cache_dir+"/models")
# 加载 Qwen2.5-VL-7B-Instruct
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_path,
    torch_dtype="auto",
    device_map="auto",
    cache_dir=cache_dir + "/models"
)


2025-12-12 17:33:13,023 - modelscope - INFO - Creating symbolic link [C:\Users\yitin\.cache\modelscope\hub\models\Qwen\Qwen2.5-VL-7B-Instruct].
2025-12-12 17:33:13,023 - modelscope - WARNING - Failed to create symbolic link C:\Users\yitin\.cache\modelscope\hub\models\Qwen\Qwen2.5-VL-7B-Instruct for C:\Users\yitin\.cache\modelscope\hub\models\Qwen\Qwen2___5-VL-7B-Instruct.
The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 5/5 [00:07<00:00,  1.51s/it]


In [3]:
def predict(messages, model):
    # 加上這個，告訴 PyTorch 不需要計算梯度
    with torch.no_grad():
        # 准备推理
        text = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        image_inputs, video_inputs = process_vision_info(messages)
        inputs = processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt",
        )
        inputs = inputs.to("cuda")

        # 生成输出
        generated_ids = model.generate(**inputs, max_new_tokens=1024)
        generated_ids_trimmed = [
            out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]
        output_text = processor.batch_decode(
            generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )

        # 釋放顯存 (可選)
        del inputs
        del generated_ids
        torch.cuda.empty_cache()

        return output_text[0]

### Evaluate without fine-tuning

In [6]:
image_path = "./data/img/burger.png"
messages = [{
    "role": "user",
    "content": [
        {
        "type": "image",
        "image": image_path
        },
        {
        "type": "text",
        "text": "Generate a nutrition report for the given food item:",
        }
    ]}]

response = predict(messages, model)
messages.append({"role": "assistant", "content": f"{response}"})
print(messages[-1]["content"])

The image shows a gourmet burger with several layers of ingredients. Here is a breakdown of the visible components:

1. **Bun**: A golden-brown hamburger bun, likely made from bread dough.
2. **Meat**: A thick, juicy beef patty that appears to be cooked to a medium-rare doneness.
3. **Cheese**: Melted cheese, possibly cheddar or a similar type, is visible on top of the patty.
4. **Onions**: Sautéed onions are layered between the patty and the cheese.
5. **Tomato**: A slice of fresh tomato is placed on top of the onions.
6. **Lettuce**: Fresh lettuce leaves are added for crunch and freshness.
7. **Mayonnaise**: A dollop of mayonnaise is spread on the top bun.

### Nutrition:
- **Calories**: The exact calorie count can vary depending on the specific ingredients used, but a typical cheeseburger like this could range from 500 to 800 calories.
- **Protein**: The beef patty provides a significant amount of protein, which is essential for muscle repair and growth.
- **Carbohydrates**: The bun

### Evaluate with LoRA fine-tuned weights

In [4]:
# ====================测试模式===================
# 配置测试参数
val_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    inference_mode=True,  # 测试模式
    r=16,  # Lora 秩
    lora_alpha=32,  # Lora alaph，具体作用参见 Lora 原理
    lora_dropout=0.1,  # Dropout 比例
    bias="none",
)
# 获取测试模型
val_peft_model = PeftModel.from_pretrained(model, model_id="./output/Qwen2.5-VL-7B-nutrition/checkpoint-215", config=val_config)
val_peft_model.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2_5_VLForConditionalGeneration(
      (model): Qwen2_5_VLModel(
        (visual): Qwen2_5_VisionTransformerPretrainedModel(
          (patch_embed): Qwen2_5_VisionPatchEmbed(
            (proj): Conv3d(3, 1280, kernel_size=(2, 14, 14), stride=(2, 14, 14), bias=False)
          )
          (rotary_pos_emb): Qwen2_5_VisionRotaryEmbedding()
          (blocks): ModuleList(
            (0-31): 32 x Qwen2_5_VLVisionBlock(
              (norm1): Qwen2RMSNorm((1280,), eps=1e-06)
              (norm2): Qwen2RMSNorm((1280,), eps=1e-06)
              (attn): Qwen2_5_VLVisionAttention(
                (qkv): Linear(in_features=1280, out_features=3840, bias=True)
                (proj): Linear(in_features=1280, out_features=1280, bias=True)
              )
              (mlp): Qwen2_5_VLMLP(
                (gate_proj): lora.Linear(
                  (base_layer): Linear(in_features=1280, out_features=3420, bias=True)
               

In [8]:
image_path = "./data/img/burger.png"
messages = [{
    "role": "user",
    "content": [
        {
        "type": "image",
        "image": image_path
        },
        {
        "type": "text",
        "text": "Generate a nutrition report for the given food item:",
        }
    ]}]

response = predict(messages, val_peft_model)
messages.append({"role": "assistant", "content": f"{response}"})
print(messages[-1]["content"])

Name: Cheeseburger with Bacon and Pickles

Main Ingredients: cheeseburgers (201.0g), pickles (35.0g), onions (28.0g)

Nutritional Information (per serving):
- Calories: 649.7 kcal
- Protein: 32.4 g
- Fat: 37.1 g
- Carbohydrates: 32.9 g
- Total Mass: 264.0 g

Nutritional Analysis: This dish is relatively high in calories and fat, primarily from the cheeseburger patty and bacon. The protein content is moderate. Consider the sodium content due to the pickles and bacon, especially for individuals monitoring their sodium intake.



In [15]:
image_path = "./data/img/macarmoons.jpg"
messages = [{
    "role": "user",
    "content": [
        {
        "type": "image",
        "image": image_path
        },
        {
        "type": "text",
        "text": "Generate a nutrition report for the given food item:",
        }
    ]}]

response = predict(messages, val_peft_model)
messages.append({"role": "assistant", "content": f"{response}"})
print(messages[-1]["content"])

Name: Macaron (Single Serving)

Main Ingredients: macarons (58.0g)

Nutritional Information (per serving):
- Calories: 241.3 kcal
- Protein: 2.6 g
- Fat: 14.9 g
- Carbohydrates: 27.0 g
- Total Mass: 58.0 g

Nutritional Analysis: This macaron serving is relatively high in fat and carbohydrates, contributing significantly to its caloric content. The protein-to-calorie ratio is low. Consider portion control due to the high calorie density.



In [16]:
image_path = "./data/img/cheesecake.jpg"
messages = [{
    "role": "user",
    "content": [
        {
        "type": "image",
        "image": image_path
        },
        {
        "type": "text",
        "text": "Generate a nutrition report for the given food item:",
        }
    ]}]

response = predict(messages, val_peft_model)
messages.append({"role": "assistant", "content": f"{response}"})
print(messages[-1]["content"])

Name: Strawberry Cheesecake with Berry Sauce

Main Ingredients: strawberries (107.0g), cream cheese (98.0g), graham crackers (65.0g), sugar (43.0g)

Nutritional Information (per serving):
- Calories: 422.0 kcal
- Protein: 7.6 g
- Fat: 20.2 g
- Carbohydrates: 49.9 g
- Total Mass: 313.0 g

Nutritional Analysis: This dessert is relatively high in calories and carbohydrates, primarily from sugars and refined grains. The protein content is modest compared to the caloric value. Consider portion control due to the high carbohydrate and fat content.



In [17]:
image_path = "./data/img/pasta.jpg"
messages = [{
    "role": "user",
    "content": [
        {
        "type": "image",
        "image": image_path
        },
        {
        "type": "text",
        "text": "Generate a nutrition report for the given food item:",
        }
    ]}]

response = predict(messages, val_peft_model)
messages.append({"role": "assistant", "content": f"{response}"})
print(messages[-1]["content"])

Name: Penne with Tomato Basil Sauce

Main Ingredients: penne pasta (305.0g), tomato sauce (126.0g), parmesan cheese (49.0g)

Nutritional Information (per serving):
- Calories: 783.5 kcal
- Protein: 27.8 g
- Fat: 24.3 g
- Carbohydrates: 112.9 g
- Total Mass: 480.0 g

Nutritional Analysis: This dish provides a moderate amount of protein relative to its caloric content. The carbohydrate content is significant, primarily from pasta and wheat berry. Consider the sodium content due to the salt and tomato sauce.



In [18]:
image_path = "./data/img/milkshake.jpg"
messages = [{
    "role": "user",
    "content": [
        {
        "type": "image",
        "image": image_path
        },
        {
        "type": "text",
        "text": "Generate a nutrition report for the given food item:",
        }
    ]}]

response = predict(messages, val_peft_model)
messages.append({"role": "assistant", "content": f"{response}"})
print(messages[-1]["content"])

Name: Oreo Milkshake

Main Ingredients: milkshakes (458.0g), oreos (123.0g)

Nutritional Information (per serving):
- Calories: 769.0 kcal
- Protein: 19.6 g
- Fat: 39.4 g
- Carbohydrates: 89.0 g
- Total Mass: 581.0 g

Nutritional Analysis: This milkshake is high in calories and fat, with a moderate protein content relative to its caloric value. The carbohydrate content is also significant, likely derived from sugars and the oreo cookies. Consider portion control due to the high calorie density.



### Evaluate in OOD scenarios

In [12]:
image_path = "./data/img/fuqifeipian.jpg"
messages = [{
    "role": "user",
    "content": [
        {
        "type": "image",
        "image": image_path
        },
        {
        "type": "text",
        "text": "Generate a nutrition report for the given food item:",
        }
    ]}]

response = predict(messages, val_peft_model)
messages.append({"role": "assistant", "content": f"{response}"})
print(messages[-1]["content"])

Name: Beef and Pork Noodle Bowl with Vegetables

Main Ingredients: beef (148.0g), pork (97.0g), noodles (56.0g), cabbage (23.0g), broccoli (14.0g)

Nutritional Information (per serving):
- Calories: 814.0 kcal
- Protein: 65.0 g
- Fat: 43.0 g
- Carbohydrates: 23.0 g
- Total Mass: 352.0 g

Nutritional Analysis: This dish is high in protein, primarily from beef and pork, contributing significantly to its caloric content. The fat content is also relatively high. Fiber is present from vegetables like cabbage and broccoli, but the carbohydrate content is modest.



In [13]:
image_path = "./data/img/laziji.jpg"
messages = [{
    "role": "user",
    "content": [
        {
        "type": "image",
        "image": image_path
        },
        {
        "type": "text",
        "text": "Generate a nutrition report for the given food item:",
        }
    ]}]

response = predict(messages, val_peft_model)
messages.append({"role": "assistant", "content": f"{response}"})
print(messages[-1]["content"])

Name: Chicken with Chilies and Sesame Seeds

Main Ingredients: chicken (147.0g), chili peppers (136.0g), sesame seeds (28.0g)

Nutritional Information (per serving):
- Calories: 509.4 kcal
- Protein: 45.1 g
- Fat: 27.6 g
- Carbohydrates: 10.1 g
- Total Mass: 311.0 g

Nutritional Analysis: This dish is high in protein, providing a significant portion of the daily recommended intake per serving. The fat content is moderate, primarily derived from the chicken and sesame seeds. The carbohydrate content is relatively low, with potential sources including rice and vegetables.



In [12]:
image_path = "./data/img/cats.jpeg"
messages = [{
    "role": "user",
    "content": [
        {
        "type": "image",
        "image": image_path
        },
        {
        "type": "text",
        "text": "Generate a nutrition report for the given food item:",
        }
    ]}]

response = predict(messages, val_peft_model)
messages.append({"role": "assistant", "content": f"{response}"})
print(messages[-1]["content"])

{'role': 'assistant', 'content': 'Name: Chicken and Rice Bowl with Vegetables\n\nMain Ingredients: chicken (102.0g), white rice (54.0g), broccoli (36.9g), carrot (36.9g)\n\nNutritional Information (per serving):\n- Calories: 278.1 kcal\n- Protein: 25.6 g\n- Fat: 10.6 g\n- Carbohydrates: 24.2 g\n- Total Mass: 268.0 g\n\nNutritional Analysis: This dish provides a good source of protein relative to its caloric content. The fat content is moderate, while carbohydrates are derived from rice and vegetables. Consider increasing fiber intake by adding more non-starchy vegetables.\n'}


In [19]:
image_path = "./data/img/cats.jpeg"
messages = [{
    "role": "user",
    "content": [
        {
        "type": "image",
        "image": image_path
        },
        {
        "type": "text",
        "text": "Describe the image",
        }
    ]}]

response = predict(messages, val_peft_model)
messages.append({"role": "assistant", "content": f"{response}"})
print(messages[-1]["content"])

{'role': 'assistant', 'content': "The image shows six kittens with a light brown and white coat, sitting closely together on a soft surface. They all have large, expressive eyes that appear to be looking directly at the camera. The kittens' fur is fluffy, and their ears are perked up, giving them an alert appearance. The background is simple and does not distract from the kittens. The overall mood of the image is warm and endearing, highlighting the cuteness of the kittens."}


### Evaluate in general task

In [19]:
image_path = "./data/img/milkshake.jpg"
messages = [{
    "role": "user",
    "content": [
        {
        "type": "image",
        "image": image_path
        },
        {
        "type": "text",
        "text": "Describe the image:",
        }
    ]}]

response = predict(messages, val_peft_model)
messages.append({"role": "assistant", "content": f"{response}"})
print(messages[-1]["content"])

The image shows two tall glasses filled with a creamy, white milkshake. Each glass is topped with a generous dollop of whipped cream and sprinkled with crushed Oreo cookies. A whole Oreo cookie is placed on top of each whipped cream dollop. The glasses are garnished with black straws, and the milkshakes are served on a marble surface. The milkshakes appear to be thick and rich, with visible specks of cookie pieces throughout the creamy texture. The overall presentation is visually appealing and indulgent.


In [20]:
image_path = "./data/img/milkshake.jpg"
messages = [{
    "role": "user",
    "content": [
        {
        "type": "image",
        "image": image_path
        },
        {
        "type": "text",
        "text": "Write a quick sort in python:",
        }
    ]}]

response = predict(messages, val_peft_model)
messages.append({"role": "assistant", "content": f"{response}"})
print(messages[-1]["content"])

```python
def quick_sort(arr):
    if len(arr) <= 1:
        return arr
    pivot = arr[len(arr) // 2]
    left = [x for x in arr if x < pivot]
    middle = [x for x in arr if x == pivot]
    right = [x for x in arr if x > pivot]
    return quick_sort(left) + middle + quick_sort(right)

# Example usage
arr = [3, 6, 8, 10, 1, 2, 1]
sorted_arr = quick_sort(arr)
print(sorted_arr)
```


Load epoch 4 checkpoints

In [8]:
import gc
import torch

def flush():
    gc.collect()
    torch.cuda.empty_cache()

# 1. 嘗試把模型移到 CPU (這一步很重要!)
try:
    if 'model' in globals():
        model.to('cpu')
    if 'val_peft_model' in globals():
        val_peft_model.to('cpu')
except:
    pass

# 2. 刪除變量
del model
del val_peft_model

# 3. 清理
flush()


In [9]:
# reload the model
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_path,
    torch_dtype="auto",
    device_map="auto",
    cache_dir=cache_dir + "/models"
)

# 配置测试参数
val_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    inference_mode=True,  # 测试模式
    r=16,  # Lora 秩
    lora_alpha=32,  # Lora alaph，具体作用参见 Lora 原理
    lora_dropout=0.1,  # Dropout 比例
    bias="none",
)
# 获取测试模型
val_peft_model = PeftModel.from_pretrained(model, model_id="./output/Qwen2.5-VL-7B-nutrition/checkpoint-860", config=val_config)
val_peft_model.eval()

Loading checkpoint shards: 100%|██████████| 5/5 [00:12<00:00,  2.41s/it]


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2_5_VLForConditionalGeneration(
      (model): Qwen2_5_VLModel(
        (visual): Qwen2_5_VisionTransformerPretrainedModel(
          (patch_embed): Qwen2_5_VisionPatchEmbed(
            (proj): Conv3d(3, 1280, kernel_size=(2, 14, 14), stride=(2, 14, 14), bias=False)
          )
          (rotary_pos_emb): Qwen2_5_VisionRotaryEmbedding()
          (blocks): ModuleList(
            (0-31): 32 x Qwen2_5_VLVisionBlock(
              (norm1): Qwen2RMSNorm((1280,), eps=1e-06)
              (norm2): Qwen2RMSNorm((1280,), eps=1e-06)
              (attn): Qwen2_5_VLVisionAttention(
                (qkv): Linear(in_features=1280, out_features=3840, bias=True)
                (proj): Linear(in_features=1280, out_features=1280, bias=True)
              )
              (mlp): Qwen2_5_VLMLP(
                (gate_proj): lora.Linear(
                  (base_layer): Linear(in_features=1280, out_features=3420, bias=True)
               

In [13]:
image_path = "./data/img/salad.jpg"
messages = [{
    "role": "user",
    "content": [
        {
        "type": "image",
        "image": image_path
        },
        {
        "type": "text",
        "text": "Write a quick sort in python:",
        }
    ]}]

response = predict(messages, val_peft_model)
messages.append({"role": "assistant", "content": f"{response}"})
print(messages[-1]["content"])

Name: Mediterranean Salad with Feta and Olives

Main Ingredients: cherry tomatoes (142.0g), cucumbers (98.0g), olives (87.0g), lettuce (65.0g), onions (63.0g)

Nutritional Information (per serving):
- Calories: 251.2 kcal
- Protein: 7.4 g
- Fat: 16.4 g
- Carbohydrates: 22.1 g
- Total Mass: 545.0 g

Nutritional Analysis: This salad provides a moderate amount of calories, primarily from fat and carbohydrates. The protein content is relatively low compared to the caloric value. Consider adding a lean protein source to enhance satiety and improve the protein-to-calorie ratio.

